# Wireless Communication Networks and Systems
## LAB 04: Error Detection & Coding
*Python simulation*

---
### Learning Objectives
By the end of this lab, students will be able to:
- Explain the purpose of redundancy for error detection and (basic) correction.
- Implement and test: single parity bit, byte-wise checksum, and CRC (CRC-8 & CRC-16).
- Simulate a noisy binary channel and measure detection performance (detected vs. undetected errors).
- (Optional) Encode/decode & correct single-bit errors with Hamming(7,4).

### Pedagogical Context (lightweight)
We stay practical: build redundancy bits from scratch, flip bits in a simulated channel, then verify frames at the receiver. Focus is on detection capability (what patterns of errors are caught or missed) rather than heavy math.

### Tools
Python 3.x with `numpy` (only for RNG convenience) and `matplotlib` (optional for plots).


---
## Step 0 — Starter Cell: Imports & Helpers
Paste and run this once at the top of your notebook/script.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(123)

# --- Bit/byte utilities ---
def bytes_to_bits(b: bytes):
    return [(byte >> i) & 1 for byte in b for i in range(7, -1, -1)]


def bits_to_bytes(bits):
    # pad to multiple of 8
    if len(bits) % 8 != 0:
        bits = bits + [0] * (8 - len(bits) % 8)
    out = bytearray()
    for i in range(0, len(bits), 8):
        byte = 0
        for j, bit in enumerate(bits[i : i + 8]):
            byte |= (bit & 1) << (7 - j)
        out.append(byte)
    return bytes(out)


def flip_bit(x: int, i: int) -> int:
    """Flip bit i (0=LSB) of integer x."""
    return x ^ (1 << i)


# --- Channel models ---
def bsc(bits, p, rng=rng):
    """Binary Symmetric Channel: flips each bit with prob p."""
    flips = rng.random(len(bits)) < p
    return [(b ^ int(f)) for b, f in zip(bits, flips)]


def burst_channel(bits, burst_len=8, burst_prob=0.05, rng=rng):
    """Simple burst: with prob, pick start and flip burst_len consecutive bits."""
    out = bits[:]
    if rng.random() < burst_prob and len(bits) >= burst_len:
        start = rng.integers(0, len(bits) - burst_len + 1)
        for i in range(start, start + burst_len):
            out[i] ^= 1
    return out


# --- Framing ---
def make_payload(n_bytes=32, mode="random", rng=rng):
    if mode == "random":
        return bytes(rng.integers(0, 256, size=n_bytes, dtype=np.uint8))
    if mode == "text":
        s = "Hello wireless students! "  # will be truncated/padded
        data = (s * ((n_bytes // len(s)) + 1)).encode()[:n_bytes]
        return data
    raise ValueError("unknown mode")


---
## Step 1 — Parity Bit (Even/Odd)

**Idea:** Append 1 bit so total number of 1s is even (even parity) or odd (odd parity). Detects any odd number of bit flips in protected block.

### 1A. Byte-wise Even Parity


In [2]:
def even_parity_bit(byte: int) -> int:
    """Return parity bit (0/1) such that total ones (byte+parity) is even."""
    ones = bin(byte).count("1")
    return ones & 1  # 1 if odd, 0 if even


def add_parity_frame(payload: bytes, parity="even"):
    bits = []
    for b in payload:
        # 8 data bits
        for i in range(7, -1, -1):
            bits.append((b >> i) & 1)
        # parity bit
        p = even_parity_bit(b)
        if parity == "even":
            bits.append(p)  # ensures even total ones
        else:
            bits.append(p ^ 1)  # odd parity if requested
    return bits


def check_parity_frame(bits, parity="even"):
    assert len(bits) % 9 == 0
    ok = True
    for i in range(0, len(bits), 9):
        data = bits[i : i + 8]
        pbit = bits[i + 8]
        ones = sum(data) + pbit
        if parity == "even":
            ok &= (ones % 2 == 0)
        else:
            ok &= (ones % 2 == 1)
    return ok


### 1B. Quick Test


In [3]:
pl = make_payload(16, "text")
frame = add_parity_frame(pl, parity="even")
# Send through BSC with p=0.02
rx = bsc(frame, p=0.02)
print("Parity check passed:", check_parity_frame(rx, "even"))


Parity check passed: False


> **Note:** Two bit flips in the same protected block may go undetected (even number of errors).

**Observation / Answer:**
Parity only checks whether the total number of 1s is even or odd:
- If **1 bit flips**, the count of 1s changes from even to odd (or odd to even), so the error is easily caught.
- If **2 bits flip** in the same byte, the number of 1s changes by +2, -2, or 0. The total stays even, so the parity check passes and the errors go completely undetected.
In general, simple parity fails to detect any **even number of bit errors** in a byte.


---
## Step 2 — Simple Checksums

**Idea:** Sum all bytes (with wrap-around), then append the sum (or its ones’ complement). Catches many errors but not all.

### 2A. 8-bit Modular Sum (Wrap to 8 bits)


In [4]:
def checksum8(payload: bytes) -> int:
    s = sum(payload) & 0xFF
    return s


def add_checksum8_frame(payload: bytes):
    cs = checksum8(payload)
    return list(bytes(payload + bytes([cs]))), cs  # return byte-level frame


def verify_checksum8_frame(frame_bytes: bytes) -> bool:
    data, cs = frame_bytes[:-1], frame_bytes[-1]
    return checksum8(data) == cs


### 2B. Ones’ Complement Checksum (Internet-style)


In [5]:
def ones_complement_add(a, b, width=16):
    mask = (1 << width) - 1
    s = a + b
    # wrap-around carry
    return (s & mask) + (s >> width)


def checksum16_ones_complement(payload: bytes) -> int:
    # pad to 16-bit words
    if len(payload) % 2 == 1:
        payload += bytes([0])
    s = 0
    for i in range(0, len(payload), 2):
        word = (payload[i] << 8) | payload[i + 1]
        s = ones_complement_add(s, word, width=16)
    return (~s) & 0xFFFF  # ones' complement


def add_checksum16_frame(payload: bytes):
    cs = checksum16_ones_complement(payload)
    frame = payload + bytes([cs >> 8, cs & 0xFF])
    return list(frame), cs


def verify_checksum16_frame(frame_bytes: bytes) -> bool:
    data, csH, csL = frame_bytes[:-2], frame_bytes[-2], frame_bytes[-1]
    cs = (csH << 8) | csL
    return checksum16_ones_complement(data) == cs


> **Undetected example:** Some paired changes can preserve the sum; checksums are not as strong as CRCs.

**Observation / Answer:**
Checksums simply add up the numbers, which has several weaknesses:
- **Compensating errors:** If one byte increases by an amount (e.g., bit flips 0 to 1) and another byte decreases by that same amount (bit flips 1 to 0), the two errors cancel out and the sum stays the same.
- **Order doesn't matter:** Because addition is commutative ($A + B = B + A$), swapping the position of two bytes does not change the checksum at all.
- Adding zero bytes or flipping all bits between +0 and -0 in ones' complement also goes undetected.


---
## Step 3 — CRC (Cyclic Redundancy Check)

**Idea:** Polynomial division over $\text{GF}(2)$. Strong burst-error detection; widely used (Ethernet, LTE/5G channels use variants).

### 3A. Generic Bitwise CRC Engine


In [6]:
def crc_bits(data_bits, poly, init=0, width=8, refin=False, refout=False, xorout=0):
    """
    Simple bitwise CRC on bits (MSB-first if refin=False).
    poly: generator polynomial without the top bit (e.g., 0x07 for CRC-8 x^8+x^2+x+1)
    width: CRC width in bits
    """
    reg = init
    topbit = 1 << (width - 1)
    for b in data_bits:
        bit = b & 1
        if refin:
            # LSB-first input
            carry = (reg & 1) ^ bit
            reg >>= 1
            if carry:
                reg ^= (poly << (width - 8))  # assumes poly fits 8 bits for width>=8
        else:
            # MSB-first input
            carry = ((reg & topbit) != 0) ^ (bit != 0)
            reg = ((reg << 1) & ((1 << width) - 1))
            if carry:
                reg ^= poly
    if refout:
        # reflect output if required (not used here)
        out = 0
        for i in range(width):
            out |= ((reg >> i) & 1) << (width - 1 - i)
        reg = out
    return reg ^ xorout


# Convenience wrappers for two common CRCs (simple params)
def crc8(data_bits):
    # CRC-8 poly x^8 + x^2 + x + 1 -> 0x07 (MSB-first)
    return crc_bits(data_bits, poly=0x07, init=0x00, width=8)


def crc16_ccitt(data_bits):
    # CRC-16-CCITT (X^16 + X^12 + X^5 + 1): 0x1021 (MSB-first), init 0xFFFF common
    return crc_bits(data_bits, poly=0x1021, init=0xFFFF, width=16)


### 3B. Framing with CRC


In [7]:
def add_crc8_frame(payload: bytes):
    data_bits = bytes_to_bits(payload)
    c = crc8(data_bits)
    crc_bits8 = [(c >> i) & 1 for i in range(7, -1, -1)]
    frame_bits = data_bits + crc_bits8
    return frame_bits, c


def verify_crc8_frame(frame_bits):
    data = frame_bits[:-8]
    recv_crc_bits = frame_bits[-8:]
    recv_crc = 0
    for i, b in enumerate(recv_crc_bits):
        recv_crc |= (b & 1) << (7 - i)
    return crc8(data) == recv_crc


def add_crc16_frame(payload: bytes):
    data_bits = bytes_to_bits(payload)
    c = crc16_ccitt(data_bits)
    crc_bits16 = [(c >> i) & 1 for i in range(15, -1, -1)]
    frame_bits = data_bits + crc_bits16
    return frame_bits, c


def verify_crc16_frame(frame_bits):
    data = frame_bits[:-16]
    recv_crc_bits = frame_bits[-16:]
    recv_crc = 0
    for i, b in enumerate(recv_crc_bits):
        recv_crc |= (b & 1) << (15 - i)
    return crc16_ccitt(data) == recv_crc


> **Facts (to discuss):** CRC-16 detects all single-bit, double-bit errors, any odd number of bit errors (for some polynomials), and all bursts shorter than its degree (here up to 16 bits).

**Observation / Answer:**
CRC treats bits as polynomials and uses division over GF(2) instead of simple addition, giving strong mathematical guarantees:
- **Single and double bit errors:** The generator polynomial is chosen so that 1-bit and 2-bit error patterns can never divide evenly into it, so they are always caught.
- **Any odd number of errors:** Caught because the polynomial has $(x+1)$ as a factor.
- **Burst errors:** Any burst of errors up to the degree of the polynomial (up to 8 bits for CRC-8, and up to 16 bits for CRC-16) is 100% guaranteed to be detected.


---
## Step 4 — Monte Carlo: Detection Performance

Run many frames through noisy channel; measure detection rates for each scheme.

### 4A. Experiment Harness


In [8]:
def run_trial(payload_len=128, p=0.01, scheme="parity", channel="bsc", burst_len=8, rng=rng):
    payload = make_payload(payload_len, "random", rng)
    if scheme == "parity":
        tx_bits = add_parity_frame(payload, "even")
        if channel == "bsc":
            rx_bits = bsc(tx_bits, p, rng)
        else:
            rx_bits = burst_channel(tx_bits, burst_len=burst_len, burst_prob=0.5, rng=rng)
        ok = check_parity_frame(rx_bits, "even")
        err = (tx_bits != rx_bits)
        return ok, err
    elif scheme == "cs8":
        tx = bytes(payload + bytes([checksum8(payload)]))
        tx_bits = bytes_to_bits(tx)
        if channel == "bsc":
            rx_bits = bsc(tx_bits, p, rng)
        else:
            rx_bits = burst_channel(tx_bits, burst_len=burst_len, burst_prob=0.5, rng=rng)
        rx = bits_to_bytes(rx_bits)
        ok = verify_checksum8_frame(rx)
        err = (tx != rx)
        return ok, err
    elif scheme == "cs16":
        frame, _ = add_checksum16_frame(payload)
        tx = bytes(frame)
        tx_bits = bytes_to_bits(tx)
        if channel == "bsc":
            rx_bits = bsc(tx_bits, p, rng)
        else:
            rx_bits = burst_channel(tx_bits, burst_len=burst_len, burst_prob=0.5, rng=rng)
        rx = bits_to_bytes(rx_bits)
        ok = verify_checksum16_frame(rx)
        err = (tx != rx)
        return ok, err
    elif scheme == "crc8":
        tx_bits, _ = add_crc8_frame(payload)
        if channel == "bsc":
            rx_bits = bsc(tx_bits, p, rng)
        else:
            rx_bits = burst_channel(tx_bits, burst_len=burst_len, burst_prob=0.5, rng=rng)
        ok = verify_crc8_frame(rx_bits)
        err = (tx_bits != rx_bits)
        return ok, err
    elif scheme == "crc16":
        tx_bits, _ = add_crc16_frame(payload)
        if channel == "bsc":
            rx_bits = bsc(tx_bits, p, rng)
        else:
            rx_bits = burst_channel(tx_bits, burst_len=burst_len, burst_prob=0.5, rng=rng)
        ok = verify_crc16_frame(rx_bits)
        err = (tx_bits != rx_bits)
        return ok, err
    else:
        raise ValueError("unknown scheme")


def evaluate(Ntrials=300, payload_len=128, p=0.01, channel="bsc", burst_len=8):
    schemes = ["parity", "cs8", "cs16", "crc8", "crc16"]
    results = {}
    for sch in schemes:
        detected = undetected = clean = 0
        for _ in range(Ntrials):
            ok_flag, any_error = run_trial(payload_len, p, sch, channel, burst_len, rng)
            if not any_error:
                clean += 1
            else:
                if ok_flag:
                    undetected += 1
                else:
                    detected += 1
        results[sch] = {
            "trials": Ntrials,
            "clean_frames": clean,
            "errored_frames": Ntrials - clean,
            "detected_errors": detected,
            "undetected_errors": undetected,
            "undetected_rate": undetected / max(1, (Ntrials - clean)),
        }
    return results


res = evaluate(Ntrials=300, payload_len=128, p=0.01)
for k, v in res.items():
    print(f"{k:8s}: {v}")


parity  : {'trials': 300, 'clean_frames': 0, 'errored_frames': 300, 'detected_errors': 300, 'undetected_errors': 0, 'undetected_rate': 0.0}
cs8     : {'trials': 300, 'clean_frames': 0, 'errored_frames': 300, 'detected_errors': 299, 'undetected_errors': 1, 'undetected_rate': 0.0033333333333333335}
cs16    : {'trials': 300, 'clean_frames': 0, 'errored_frames': 300, 'detected_errors': 300, 'undetected_errors': 0, 'undetected_rate': 0.0}
crc8    : {'trials': 300, 'clean_frames': 0, 'errored_frames': 300, 'detected_errors': 300, 'undetected_errors': 0, 'undetected_rate': 0.0}
crc16   : {'trials': 300, 'clean_frames': 0, 'errored_frames': 300, 'detected_errors': 300, 'undetected_errors': 0, 'undetected_rate': 0.0}


### 4B. Suggested Variations
- Change $p$ (e.g., $0.001$, $0.01$, $0.05$) and compare `undetected_rate`.
- Replace BSC by `burst_channel` and discuss which scheme better handles bursts (CRC wins).


In [9]:
# Variation 1: Compare undetected rates across different BSC error probabilities (N=2000, payload=128 bytes)
p_vals = [0.001, 0.01, 0.05]
schemes = ["parity", "cs8", "cs16", "crc8", "crc16"]

print("=== Channel: bsc | N=2000 | payload=128 bytes ===")
print(f"{'p':>8s}  {'scheme':>6s}  {'clean':>5s}  {'errored':>7s}  {'detected':>8s}  {'undetected':>10s}  {'undet_rate':>10s}")

bsc_results = []
for p in p_vals:
    for sch in schemes:
        clean = errored = detected = undetected = 0
        for _ in range(2000):
            ok, err = run_trial(payload_len=128, p=p, scheme=sch, channel="bsc", rng=rng)
            if not err:
                clean += 1
            else:
                errored += 1
                if ok:
                    undetected += 1
                else:
                    detected += 1
        undet_rate = undetected / max(1, errored)
        print(f"{p:8.3f}  {sch:>6s}  {clean:5d}  {errored:7d}  {detected:8d}  {undetected:10d}  {undet_rate:10.4f}")
        bsc_results.append({
            "p": p,
            "scheme": sch,
            "clean": clean,
            "errored": errored,
            "detected": detected,
            "undetected": undetected,
            "undet_rate": undet_rate
        })


=== Channel: bsc | N=2000 | payload=128 bytes ===
       p  scheme  clean  errored  detected  undetected  undet_rate
   0.001  parity    663     1337      1332           5      0.0037
   0.001     cs8    724     1276      1248          28      0.0219
   0.001    cs16    694     1306      1297           9      0.0069
   0.001    crc8    695     1305      1300           5      0.0038
   0.001   crc16    743     1257      1257           0      0.0000
   0.010  parity      0     2000      2000           0      0.0000
   0.010     cs8      0     2000      1989          11      0.0055
   0.010    cs16      0     2000      1999           1      0.0005
   0.010    crc8      0     2000      1993           7      0.0035
   0.010   crc16      0     2000      2000           0      0.0000
   0.050  parity      0     2000      2000           0      0.0000
   0.050     cs8      0     2000      1997           3      0.0015
   0.050    cs16      0     2000      2000           0      0.0000
   0.050    

In [10]:
# Variation 2: Performance under burst channel errors (N=2000, payload=128 bytes, burst_len=8)
print("=== Channel: burst | N=2000 | payload=128 bytes ===")
print(f"{'p':>8s}  {'scheme':>6s}  {'clean':>5s}  {'errored':>7s}  {'detected':>8s}  {'undetected':>10s}  {'undet_rate':>10s}")

burst_results = []
for p in p_vals:
    for sch in schemes:
        clean = errored = detected = undetected = 0
        for _ in range(2000):
            ok, err = run_trial(payload_len=128, p=p, scheme=sch, channel="burst", burst_len=8, rng=rng)
            if not err:
                clean += 1
            else:
                errored += 1
                if ok:
                    undetected += 1
                else:
                    detected += 1
        undet_rate = undetected / max(1, errored)
        print(f"{p:8.3f}  {sch:>6s}  {clean:5d}  {errored:7d}  {detected:8d}  {undetected:10d}  {undet_rate:10.4f}")
        burst_results.append({
            "p": p,
            "scheme": sch,
            "clean": clean,
            "errored": errored,
            "detected": detected,
            "undetected": undetected,
            "undet_rate": undet_rate
        })


=== Channel: burst | N=2000 | payload=128 bytes ===
       p  scheme  clean  errored  detected  undetected  undet_rate
   0.001  parity   1015      985       441         544      0.5523
   0.001     cs8   1016      984       984           0      0.0000
   0.001    cs16    995     1005      1005           0      0.0000
   0.001    crc8   1026      974       974           0      0.0000
   0.001   crc16   1035      965       965           0      0.0000
   0.010  parity   1025      975       419         556      0.5703
   0.010     cs8   1013      987       987           0      0.0000
   0.010    cs16   1016      984       984           0      0.0000
   0.010    crc8   1010      990       990           0      0.0000
   0.010   crc16    986     1014      1014           0      0.0000
   0.050  parity   1009      991       434         557      0.5621
   0.050     cs8   1002      998       998           0      0.0000
   0.050    cs16   1010      990       990           0      0.0000
   0.050  

**Observation / Answer:**
### Comparing the Five Methods: BSC vs. Burst Channel

#### 1. Binary Symmetric Channel (BSC - Random Bit Errors)
- **At low error rate ($p = 0.001$):**
  - Around 35% of the 128-byte frames are clean, and 65% have errors.
  - **Parity** detects almost all errors (undetected rate ~0.37%), only missing when 2 errors hit the exact same byte.
  - **CS8** has the worst detection (~2.2% undetected) because random bit flips across 128 bytes easily add up to multiples of 256.
  - **CS16** performs much better (~0.7% undetected) thanks to its larger 16-bit sum space.
  - **CRC-8** is much better than CS8 (~0.4% undetected).
  - **CRC-16 is the best**, catching 100% of errors (**0 undetected errors**) across all trials.
- **At higher error rates ($p = 0.01, 0.05$):**
  - Every frame contains errors. Parity catches them all because with errors spread across 128 bytes, at least one byte is sure to have an odd number of flips.
  - CS8 still misses about 0.3% to 0.5% of errors due to random checksum collisions (roughly $1/256$).
  - CRC-16 catches 100% of errors in all tests.

#### 2. Burst Channel (8-bit Burst Errors)
- **Parity fails completely:** Over **55% of burst errors go undetected**!
  - *Why:* An 8-bit burst flips 8 bits in a row. When this hits an 8-bit byte, all 8 bits flip. Since 8 is an even number, the total number of 1s keeps the same parity (even stays even), completely fooling the parity check.
- **Checksums and CRCs catch 100% of burst errors:**
  - `cs8`, `cs16`, `crc8`, and `crc16` detected every single burst error (0 undetected).
  - For CRC-8 and CRC-16, this is mathematically guaranteed: any burst of length $\le 8$ is always caught by an 8-bit or 16-bit CRC.


---
## Step 5 — (Optional) Hamming(7,4) Single-Bit Correction

**Idea:** Encode 4 data bits into 7 with parity checks that allow correction of any single-bit error in a codeword.

### 5A. Encoder/Decoder


In [11]:
# Positions: [1..7] with parity bits at 1,2,4 (classic convention)
P = [1, 2, 4]
D = [3, 5, 6, 7]


def hamming74_encode(nibble4):  # nibble4: list of 4 bits [d1,d2,d3,d4]
    code = [0] * 7
    code[2], code[4], code[5], code[6] = nibble4  # positions 3,5,6,7
    # parity bits (even)
    code[0] = (code[2] ^ code[4] ^ code[6]) & 1  # p1 covers 1,3,5,7
    code[1] = (code[2] ^ code[5] ^ code[6]) & 1  # p2 covers 2,3,6,7
    code[3] = (code[4] ^ code[5] ^ code[6]) & 1  # p4 covers 4,5,6,7
    return code


def hamming74_syndrome(code7):
    s1 = (code7[0] ^ code7[2] ^ code7[4] ^ code7[6]) & 1
    s2 = (code7[1] ^ code7[2] ^ code7[5] ^ code7[6]) & 1
    s4 = (code7[3] ^ code7[4] ^ code7[5] ^ code7[6]) & 1
    return (s4 << 2) | (s2 << 1) | s1


def hamming74_decode(code7):
    syn = hamming74_syndrome(code7)
    corr = code7[:]
    if syn != 0 and 1 <= syn <= 7:
        corr[syn - 1] ^= 1  # correct single-bit error
    data = [corr[2], corr[4], corr[5], corr[6]]
    return data, syn


### 5B. Quick Demo


In [12]:
d = [1, 0, 1, 1]
c = hamming74_encode(d)
# flip one random bit
pos = rng.integers(0, 7)
c_err = c[:]
c_err[pos] ^= 1
d_hat, syn = hamming74_decode(c_err)
print("orig:", d, "flip_pos:", pos + 1, "syndrome:", syn, "decoded:", d_hat)


orig: [1, 0, 1, 1] flip_pos: 2 syndrome: 2 decoded: [1, 0, 1, 1]


> **Note:** $\text{Hamming}(7,4)$ corrects any single-bit error but only detects (not correct) most double-bit errors.

**Observation / Answer:**
Hamming(7,4) adds 3 parity bits to 4 data bits:
- If **1 bit flips**, the receiver calculates the syndrome, which gives the exact position (1 to 7) of the flipped bit. The receiver simply inverts that bit back to fix the error.
- If **2 bits flip**, the syndrome is still non-zero (so it knows an error happened), but it points to the wrong bit position. Trying to fix it will flip a healthy bit and make things worse. Therefore, Hamming(7,4) can detect two errors, but can only correct one.


---
## Guiding Questions


### Question 1
Give an example where parity fails but CRC-16 succeeds. Why?


**Answer:**
- **Example:** Suppose we send the byte `00000000` with an even parity bit `0`. If noise flips two bits so it becomes `11000000`, the byte now has two 1s. Since 2 is even, the parity check passes and the error is completely missed.
- **Why CRC-16 succeeds:** CRC-16 treats the bits as a polynomial. Its generator polynomial is designed so that no 2-bit error pattern can divide evenly into it, leaving a remainder that always catches the error.


### Question 2
Why does a ones’ complement checksum sometimes miss certain multi-bit patterns?


**Answer:**
Ones' complement checksums miss certain patterns because addition ignores word positions and order:
1. **Cancelling errors:** If one bit flips from 0 to 1 (+1) in one word, and another bit flips from 1 to 0 (-1) in a second word, the sum doesn't change.
2. **Swapping words:** Because $A + B = B + A$, changing the order of two words gives the exact same checksum.
3. **Zeros:** In ones' complement, all-zeros (`0x0000`) and all-ones (`0xFFFF`) both equal zero, so flipping between them doesn't change the checksum.


### Question 3
How does frame length affect undetected error probability for each scheme?


**Answer:**
- **Parity:** For random noise, longer frames have more bytes, making it more likely that at least one byte has an odd number of errors and gets caught. But for burst errors, parity stays bad (~55% undetected) no matter how long the frame is.
- **Checksums:** Longer frames mean more words, so there are more chances for errors to cancel each other out. The undetected rate settles around $1/256$ for 8-bit and $1/65536$ for 16-bit.
- **CRCs:** Bursts shorter than the CRC size are always 100% caught, regardless of frame length. For random noise, CRC-16 catches 100% of double errors as long as the frame isn't excessively large, and longer random error patterns have only a tiny chance ($1/2^{16} \approx 0.0015\%$) of slipping through.


### Question 4
In practice, why do protocols combine CRC with FEC (e.g., convolutional/LDPC codes)?


**Answer:**
They do two different, complementary jobs:
1. **FEC (Forward Error Correction):** Fixes common bit errors on the spot so the system doesn't have to waste time retransmitting packets, keeping speeds high.
2. **CRC:** Acts as a safety check. If the noise is too strong and FEC fails or makes a wrong correction, CRC reliably detects the bad packet so it can be discarded and requested again.
Together, FEC provides speed, while CRC guarantees data reliability.
